# 🌿 Plant Disease Detector — Plant Disease Recognition with TensorFlow

![TensorFlow](https://img.shields.io/badge/TensorFlow-2.x-orange?logo=tensorflow)
![Keras](https://img.shields.io/badge/Keras-EfficientNetB0-red?logo=keras)
![Python](https://img.shields.io/badge/Python-3.10+-blue?logo=python)
![License](https://img.shields.io/badge/License-MIT-green)
![Status](https://img.shields.io/badge/Status-Portfolio%20Project-brightgreen)

A complete Machine Learning project presenting the construction of a
plant disease classifier from leaf photographs, using **Transfer Learning**
and **Fine Tuning** on the **EfficientNetB0** architecture (TensorFlow /
Keras).

The notebook's structure and methodology are designed to guide the reader step by step through every stage of the project, providing a clear rationale for each design decision—from data preparation to model deployment.

---

## 🗂️ Table of Contents

1. [Introduction](#wprowadzenie)
2. [Problem Definition](#problem)
3. [Dataset Description](#dataset)
4. [Library Imports](#importy)
5. [GPU Configuration](#gpu)
6. [Dataset Download](#pobieranie)
7. [Data Analysis](#analiza)
8. [Example Image Visualization](#wizualizacja)
9. [Data Augmentation](#augmentacja)
10. [Train / Validation / Test Split](#podzial)
11. [Building the EfficientNetB0 Model](#architektura)
12. [Transfer Learning](#transfer-learning)
13. [Fine Tuning](#fine-tuning)
14. [Model Training — Summary of Both Phases](#trenowanie)
15. [Evaluation](#ewaluacja)
16. [Accuracy](#accuracy)
17. [Loss](#loss)
18. [Precision](#precision)
19. [Recall](#recall)
20. [F1-score](#f1)
21. [Confusion Matrix](#confusion-matrix)
22. [Classification Report](#classification-report)
23. [Single Image Prediction](#pred-single)
24. [Multiple Image Prediction](#pred-multi)
25. [Misprediction Visualization](#bledy)
26. [Saving the Model](#zapis)
27. [Loading the Model](#wczytanie)
28. [Prediction on New Images](#nowe-zdjecia)
29. [Summary](#podsumowanie)
30. [Model Limitations](#ograniczenia)
31. [Possible Improvements](#ulepszenia)

---

<a name="wprowadzenie"></a>
## 1️⃣ Introduction

Plant diseases account for a significant share of agricultural production
losses worldwide. Early and accurate disease identification from a leaf photo
lets farmers and agronomists take swift action (e.g. choosing the right crop
protection product) before an infection spreads across an entire crop.

The goal of this project is to build a deep learning model that automatically
recognizes, from a photo of a plant leaf:

* the **plant species** (e.g. apple, tomato, grape),
* the **type of disease** (e.g. apple scab, potato blight) — or determines
  that the plant is **healthy**.


<a name="problem"></a>
## 2️⃣ Problem Definition

**Task type:** multi-class image classification — each leaf photo belongs to
exactly one of **38 classes** (a combination of plant species and a specific
disease, or a healthy state).

**Input:** a single photo of a plant leaf (RGB).

**Output:** a class label (e.g. `Tomato___Late_blight`) together with a
prediction confidence level (a probability from the `softmax` layer).

**Success metric:** accuracy and F1-score (macro) on the test set — with 38
classes and some degree of data imbalance, accuracy alone can be misleading,
so we additionally track precision, recall, and the confusion matrix for
individual classes.

**Limitations deliberately accepted at the start of the project** (expanded
in the *Model Limitations* section): the dataset's images come from
controlled laboratory conditions (uniform background, good lighting, a
single leaf), which differs from photos taken with a phone under real
field conditions.

<a name="dataset"></a>
## 3️⃣ Dataset Description

This project uses the **PlantVillage** dataset — one of the most widely used
public datasets for plant disease classification from leaf photographs.

| Feature | Value |
|---|---|
| Number of classes | 38 (plant species × condition/disease) |
| Number of images | ~54,000 |
| Source resolution | varying (rescaled to 224×224 for the model) |
| Format | color RGB images |
| Plant species | includes apple, tomato, grape, corn, potato, bell pepper, strawberry, peach |
| Photo conditions | controlled (uniform background, laboratory) |
| Source | [Kaggle — PlantVillage Dataset](https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset) |

The dataset is downloaded from Kaggle using the official `kaggle` client
(Section 6). This requires your own (free) Kaggle account and API key — see
the instructions in the *Dataset Download* section.


<a name="importy"></a>
## 4️⃣ Library Imports


In [ ]:
# ---------------------------------------------------------------
# Installation (if running in Google Colab / a fresh environment)
# ---------------------------------------------------------------
# IMPORTANT: we deliberately do NOT use --upgrade for tensorflow/pandas/scikit-learn.
# Google Colab ships with a pre-installed, mutually compatible set of these
# libraries (matched to other Colab tools, e.g. protobuf, cudf, tf-keras).
# Forcing the latest versions (`--upgrade`) can desynchronize that set and
# trigger a cascade of dependency conflicts. We only install what is actually
# missing: the Kaggle client (to download the dataset) and seaborn.
!pip install --quiet kaggle seaborn

# ---------------------------------------------------------------
# Imports
# ---------------------------------------------------------------
import os
import json
import datetime
import pathlib
import random
import zipfile
import getpass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Plot style
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

print("✅ TensorFlow version:", tf.__version__)


<a name="gpu"></a>
## 5️⃣ GPU Configuration

Training a convolutional network on ~54,000 images is computationally
demanding — using a GPU is strongly recommended (in Google Colab: `Runtime →
Change runtime type → GPU`). The cell below checks GPU availability and
configures **dynamic memory allocation** (`memory growth`) so TensorFlow
doesn't immediately reserve the entire GPU memory — good practice, especially
on a shared environment (e.g. Colab).


In [ ]:
gpus = tf.config.list_physical_devices("GPU")

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✅ Detected {len(gpus)} GPU(s). Memory growth enabled.")
        for gpu in gpus:
            print("   -", gpu)
    except RuntimeError as e:
        # set_memory_growth must be called before GPU initialization
        print("⚠️ Failed to configure memory growth:", e)
else:
    print("⚠️ No GPU detected — training will run on CPU (significantly slower).")
    print("   In Google Colab: Runtime → Change runtime type → Hardware accelerator: GPU")

# Mixed precision (optional) — speeds up training on newer GPUs (Tensor Cores),
# while maintaining numerical stability thanks to float32 for critical weights.
# Uncomment if training on a GPU that supports mixed precision (e.g. T4, A100, RTX):
# from tensorflow.keras import mixed_precision
# mixed_precision.set_global_policy("mixed_float16")
# print("✅ Mixed precision enabled:", mixed_precision.global_policy())


<a name="pobieranie"></a>
## 6️⃣ Dataset Download

We download the **PlantVillage** dataset from Kaggle using the official
`kaggle` client. This requires your own (free) API key:

1. Create an account at [kaggle.com](https://www.kaggle.com) (if you don't
   have one yet).
2. Go to **Account → API → Create New API Token** — this downloads a
   `kaggle.json` file containing your `username` and `key`.
3. Paste the values from that file below when the notebook prompts for them
   (`getpass` — the key is **never** hard-coded anywhere, so this notebook
   can be safely published on GitHub).

Since the exact folder structure inside the Kaggle archive can vary, the code
below **automatically detects** the correct folder with 38 class subfolders
(and among several variants — `color` / `grayscale` / `segmented` — picks the
color version, consistent with the project's assumptions).

> ℹ️ The downloaded dataset (`data/`) is cached locally — re-running this
> cell won't re-download the data.


In [ ]:
def find_data_directory(root_dir, expected_classes=38):
    # Searches the directory tree for a folder containing exactly
    # `expected_classes` subfolders (classes). If several such candidates
    # exist (e.g. color/grayscale/segmented variants), prefers the one
    # containing 'color' in its name.
    candidates = []
    for dirpath, dirnames, _ in os.walk(root_dir):
        if len(dirnames) == expected_classes:
            candidates.append(dirpath)

    if not candidates:
        raise FileNotFoundError(
            f"No folder with {expected_classes} subfolders (classes) was found in '{root_dir}'.\n"
            f"Check the downloaded data structure manually, e.g. with:\n"
            f"  !find {root_dir} -maxdepth 3 -type d\n"
            f"and set the DATA_DIR variable manually to the correct path."
        )

    color_candidates = [c for c in candidates if "color" in c.lower()]
    return color_candidates[0] if color_candidates else candidates[0]


print("🔑 Downloading the dataset requires a Kaggle API key (Account → API → Create New API Token).")
KAGGLE_USERNAME = os.environ.get("KAGGLE_USERNAME") or getpass.getpass("Enter KAGGLE_USERNAME: ")
KAGGLE_KEY = os.environ.get("KAGGLE_KEY") or getpass.getpass("Enter KAGGLE_KEY: ")
os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
os.environ["KAGGLE_KEY"] = KAGGLE_KEY

DATA_ROOT = "data"
KAGGLE_DATASET = "abdallahalidev/plantvillage-dataset"
DOWNLOAD_MARKER = os.path.join(DATA_ROOT, ".download_complete")

os.makedirs(DATA_ROOT, exist_ok=True)

if not os.path.exists(DOWNLOAD_MARKER):
    !kaggle datasets download -d {KAGGLE_DATASET} -p {DATA_ROOT} --unzip
    pathlib.Path(DOWNLOAD_MARKER).touch()
    print("✅ Download complete.")
else:
    print("✅ Dataset already downloaded — skipping re-download.")

IMG_SIZE = 224          # required input size for EfficientNetB0
BATCH_SIZE = 32

DATA_DIR = find_data_directory(DATA_ROOT, expected_classes=38)
print(f"📁 Detected data folder (38 classes): {DATA_DIR}")

class_names = sorted(
    d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))
)
NUM_CLASSES = len(class_names)
TOTAL_EXAMPLES = sum(len(os.listdir(os.path.join(DATA_DIR, c))) for c in class_names)

print(f"🔢 Number of classes: {NUM_CLASSES}")
print(f"🖼️ Number of images: {TOTAL_EXAMPLES}")
print("\nExample class names:")
for name in class_names[:5]:
    print("  -", name)
print("  ...")


<a name="analiza"></a>
## 7️⃣ Data Analysis

Before building the model, we check the **distribution of image counts
across classes**. Imbalanced data (some classes have far more examples than
others) can cause the model to "favor" more populous classes — worth
diagnosing this right from the start, to consciously decide whether
additional techniques (e.g. class weighting) are needed, or whether standard
training will suffice.

> ⚠️ A full pass through ~54,000 images to count labels can take a few
> minutes — this is a one-time exploratory analysis cost.


In [ ]:
class_counts = pd.DataFrame({
    "class": class_names,
    "image_count": [len(os.listdir(os.path.join(DATA_DIR, c))) for c in class_names],
}).sort_values("image_count", ascending=True)

print(class_counts.describe())
print(f"\nLeast populated class: {class_counts.iloc[0]['class']} ({class_counts.iloc[0]['image_count']} images)")
print(f"Most populated class:  {class_counts.iloc[-1]['class']} ({class_counts.iloc[-1]['image_count']} images)")

plt.figure(figsize=(10, 14))
plt.barh(class_counts["class"], class_counts["image_count"], color="seagreen")
plt.xlabel("Number of images")
plt.title("Image count per class — PlantVillage (38 classes)")
plt.tight_layout()
plt.show()

imbalance_ratio = class_counts["image_count"].max() / class_counts["image_count"].min()
print(f"\n📊 Imbalance ratio (max/min): {imbalance_ratio:.1f}x")


### Building the File List

Before proceeding, we build a single list containing the path and label for
**every** image in the dataset — the starting point for both visualization
(Section 8) and the formal train/validation/test split (Section 10).


In [ ]:
image_paths = []
image_labels = []

for class_idx, class_name in enumerate(class_names):
    class_dir = os.path.join(DATA_DIR, class_name)
    for fname in os.listdir(class_dir):
        image_paths.append(os.path.join(class_dir, fname))
        image_labels.append(class_idx)

image_paths = np.array(image_paths)
image_labels = np.array(image_labels)

print(f"🖼️ Total number of images: {len(image_paths)}")


<a name="wizualizacja"></a>
## 8️⃣ Example Image Visualization

Before we start processing the data numerically, it's worth seeing it "live"
— this helps build intuition about the character of the images (background,
framing, visibility of disease symptoms) and spot potential data issues.


In [ ]:
def load_image(path):
    raw = tf.io.read_file(path)
    image = tf.io.decode_image(raw, channels=3, expand_animations=False)
    return image


rng = np.random.default_rng(SEED)
sample_idx = rng.choice(len(image_paths), size=9, replace=False)

plt.figure(figsize=(12, 12))
for i, idx in enumerate(sample_idx):
    image = load_image(image_paths[idx])
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(image.numpy())
    plt.title(class_names[image_labels[idx]], fontsize=9)
    plt.axis("off")
plt.suptitle("Example images from the PlantVillage dataset", fontsize=14)
plt.tight_layout()
plt.show()


<a name="augmentacja"></a>
## 9️⃣ Data Augmentation

**Data Augmentation** is a technique for artificially increasing the
diversity of training data through random image transformations (rotation,
horizontal flip, zoom, contrast change) — without collecting new images.
This reduces the risk of overfitting and improves the model's ability to
generalize to new, previously unseen images (e.g. taken from a different
angle or under different lighting).

We implement augmentation as **Keras layers** (`tf.keras.Sequential`), which
we'll later attach directly to the model. This approach has two advantages:

1. Augmentation is active **only during training** (`training=True`) —
   automatically disabled during evaluation and prediction.
2. Augmentation becomes part of the model's computation graph, so it's
   automatically included when saving/loading the model (`model.save`).


In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.15),
    layers.RandomTranslation(0.1, 0.1),
], name="data_augmentation")

# Visualization: the same image+augmentation pair applied 9 times, to see
# the diversity of random transformations.
sample_idx_aug = rng.integers(0, len(image_paths))
sample_image = tf.image.resize(load_image(image_paths[sample_idx_aug]), (IMG_SIZE, IMG_SIZE))
sample_label = image_labels[sample_idx_aug]
sample_image_batch = tf.expand_dims(sample_image, axis=0)

plt.figure(figsize=(12, 12))
for i in range(9):
    augmented = data_augmentation(sample_image_batch, training=True)
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(augmented[0].numpy().astype("uint8"))
    plt.axis("off")
plt.suptitle(f"Data Augmentation — example: {class_names[sample_label]}", fontsize=13)
plt.tight_layout()
plt.show()


<a name="podzial"></a>
## 🔟 Train / Validation / Test Split

We split the list of all images (`image_paths` / `image_labels`) into three
disjoint parts — **explicitly and reproducibly** (fixed `SEED`), so results
can be reproduced across runs:

| Set | Share | Purpose |
|---|---|---|
| Train | 70% | training the model's weights |
| Validation | 15% | monitoring during training (early stopping, hyperparameter selection) |
| Test | 15% | final, one-time model evaluation (not used during training) |

We then build a `tf.data` pipeline for each set: reading a file from disk,
decoding, resizing to 224×224 (required EfficientNetB0 input), batching, and
`prefetch` for performance (overlapping data preparation with GPU training).


In [ ]:
indices = np.arange(len(image_paths))
rng.shuffle(indices)

n_total = len(indices)
n_train = int(n_total * 0.70)
n_val = int(n_total * 0.15)

train_idx = indices[:n_train]
val_idx = indices[n_train:n_train + n_val]
test_idx = indices[n_train + n_val:]


def make_dataset_from_indices(idx):
    return tf.data.Dataset.from_tensor_slices((image_paths[idx], image_labels[idx]))


def load_and_preprocess(path, label):
    image = tf.io.read_file(path)
    image = tf.io.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32)  # EfficientNet has built-in normalization — no manual scaling needed
    return image, label


def prepare_dataset(ds, shuffle=False, batch_size=BATCH_SIZE):
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(buffer_size=2000, seed=SEED)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


train_ds = prepare_dataset(make_dataset_from_indices(train_idx), shuffle=True)
val_ds = prepare_dataset(make_dataset_from_indices(val_idx))
test_ds = prepare_dataset(make_dataset_from_indices(test_idx))

split_sizes = {"train": len(train_idx), "validation": len(val_idx), "test": len(test_idx)}
print("📊 Split sizes:", split_sizes)

plt.figure(figsize=(6, 5))
plt.bar(split_sizes.keys(), split_sizes.values(), color=["#4C72B0", "#DD8452", "#55A868"])
plt.title("Dataset split: Train / Validation / Test")
plt.ylabel("Number of images")
for i, (k, v) in enumerate(split_sizes.items()):
    plt.text(i, v, str(v), ha="center", va="bottom")
plt.tight_layout()
plt.show()


<a name="architektura"></a>
## 1️⃣1️⃣ Building the EfficientNetB0 Model

**EfficientNetB0** is a convolutional network designed for an optimal
accuracy-to-parameter-count ratio (so-called *compound scaling* of network
depth, width, and resolution). It's a good base choice for transfer
learning: powerful enough, yet lightweight and fast to train.

**Final model architecture:**

```
Input (224, 224, 3)
        ↓
Data Augmentation (active only during training)
        ↓
EfficientNetB0 (base, ImageNet weights, initially frozen)
        ↓
GlobalAveragePooling2D
        ↓
Dropout (0.3)
        ↓
Dense (38, softmax)  ← our classification layer
```

The base layer is loaded with weights trained on ImageNet
(`weights="imagenet"`) and **without** the original classification head
(`include_top=False`) — in its place, we add our own, tailored to 38 classes.


In [ ]:
base_model = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    pooling=None,
)
base_model.trainable = False  # freeze weights during the transfer-learning stage

inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="input_layer")
x = data_augmentation(inputs)
# training=False: even after unfreezing layers during fine-tuning (Section 13),
# BatchNormalization layers will remain in inference mode — recommended Keras
# practice when fine-tuning models with BatchNorm (prevents "corrupting" the
# learned normalization statistics with small batches during fine-tuning).
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D(name="global_average_pooling")(x)
x = layers.Dropout(0.3, name="top_dropout")(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax", name="output_layer")(x)

model = tf.keras.Model(inputs, outputs, name="plant_disease_efficientnetb0")
model.summary()

print(f"\n🔢 Number of layers in the base model: {len(base_model.layers)}")
print(f"🔒 Base model frozen: {not base_model.trainable}")


<a name="transfer-learning"></a>
## 1️⃣2️⃣ Transfer Learning (Phase 1 — Training the Classification Head)

In the first training phase, EfficientNetB0's weights remain **frozen**
(`base_model.trainable = False`) — only the newly added classification head
(`GlobalAveragePooling2D → Dropout → Dense`) is trained. As a result:

* training is fast (far fewer parameters to update),
* we avoid the risk of destroying the well-trained, general visual features
  (edges, textures, shapes) learned on ImageNet.

**Callbacks used during training:**

| Callback | Role |
|---|---|
| `ModelCheckpoint` | saves the best model version (by `val_accuracy`) |
| `EarlyStopping` | stops training once the model stops improving |
| `ReduceLROnPlateau` | reduces the learning rate when training plateaus |
| `TensorBoard` | logs training progress for visualization |


In [ ]:
MODELS_DIR = "models"
LOGS_DIR = "logs"
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(LOGS_DIR, exist_ok=True)

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

phase1_callbacks = [
    callbacks.ModelCheckpoint(
        filepath=os.path.join(MODELS_DIR, "phase1_feature_extraction.keras"),
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1,
    ),
    callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True, verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.2, patience=3, min_lr=1e-7, verbose=1
    ),
    callbacks.TensorBoard(log_dir=os.path.join(LOGS_DIR, f"phase1_{timestamp}")),
]

EPOCHS_PHASE1 = 10

history_1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE1,
    callbacks=phase1_callbacks,
    verbose=1,
)


<a name="fine-tuning"></a>
## 1️⃣3️⃣ Fine Tuning (Phase 2 — Precise Adjustment of Base Layers)

After the classification head has stabilized, we **unfreeze** the last
layers of EfficientNetB0, so the model can gently adapt more specialized,
high-level visual features to the specifics of our dataset (shapes and
textures characteristic of plant disease symptoms).

**Key rules for safe fine-tuning:**

* We only unfreeze the **last N layers** (not the whole model) — earlier
  layers learn very general features (edges, colors) that don't need
  adjusting.
* We use a **much smaller learning rate** (`1e-5` instead of `1e-3`) — large
  update steps on pre-trained weights would quickly destroy the learned
  knowledge ("catastrophic forgetting").
* `BatchNormalization` layers remain in inference mode (see the comment in
  Section 11) regardless of unfreezing — this prevents instability caused by
  small batch sizes.


In [ ]:
FINE_TUNE_AT = len(base_model.layers) - 30  # unfreeze the last 30 layers

base_model.trainable = True
for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False

trainable_count = sum(1 for layer in base_model.layers if layer.trainable)
print(f"🔓 Unfrozen base-model layers: {trainable_count} / {len(base_model.layers)}")

# Recompilation is required for the `trainable` change to take effect
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

phase2_callbacks = [
    callbacks.ModelCheckpoint(
        filepath=os.path.join(MODELS_DIR, "phase2_fine_tuning.keras"),
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1,
    ),
    callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True, verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.2, patience=3, min_lr=1e-8, verbose=1
    ),
    callbacks.TensorBoard(log_dir=os.path.join(LOGS_DIR, f"phase2_{timestamp}")),
]

EPOCHS_PHASE2 = 10
TOTAL_EPOCHS = EPOCHS_PHASE1 + EPOCHS_PHASE2

history_2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=TOTAL_EPOCHS,
    initial_epoch=history_1.epoch[-1] + 1,  # continue epoch numbering from phase 1
    callbacks=phase2_callbacks,
    verbose=1,
)


<a name="trenowanie"></a>
## 1️⃣4️⃣ Model Training — Summary of Both Phases

We merge the training history from **Phase 1** (transfer learning) and
**Phase 2** (fine-tuning) into one consistent chart, with a dashed line
marking the transition between phases. This lets us assess whether
fine-tuning actually improved results compared to transfer learning alone.


In [ ]:
def merge_histories(hist1, hist2):
    merged = {}
    for key in hist1.history:
        merged[key] = hist1.history[key] + hist2.history[key]
    return merged


history = merge_histories(history_1, history_2)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(history["accuracy"], label="Train accuracy")
axes[0].plot(history["val_accuracy"], label="Validation accuracy")
axes[0].axvline(EPOCHS_PHASE1 - 1, color="gray", linestyle="--", label="Fine-tuning start")
axes[0].set_title("Accuracy — both training phases")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history["loss"], label="Train loss")
axes[1].plot(history["val_loss"], label="Validation loss")
axes[1].axvline(EPOCHS_PHASE1 - 1, color="gray", linestyle="--", label="Fine-tuning start")
axes[1].set_title("Loss — both training phases")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"📈 Best validation accuracy: {max(history['val_accuracy']):.4f}")
print(f"📉 Lowest validation loss:  {min(history['val_loss']):.4f}")


<a name="ewaluacja"></a>
## 1️⃣5️⃣ Evaluation

The final, **one-time** evaluation of the model on the test set — data the
model has never seen, either during training or during monitoring
(validation). This is the most reliable estimate of how the model will
behave on new, unseen images.


In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds, verbose=1)

print(f"\n📊 Test set results:")
print(f"   Loss:     {test_loss:.4f}")
print(f"   Accuracy: {test_accuracy:.4f} ({test_accuracy * 100:.2f}%)")


### Preparing Test-Set Predictions

The cell below is run once and used in all subsequent metric sections
(Accuracy, Precision, Recall, F1, Confusion Matrix, Classification Report)
and prediction visualizations — we collect predictions, true labels, and the
images themselves (for later error visualizations).


In [ ]:
y_true = []
y_pred = []
y_pred_probs = []
test_images = []

for images_batch, labels_batch in test_ds:
    preds_batch = model.predict(images_batch, verbose=0)
    y_pred_probs.extend(preds_batch)
    y_pred.extend(np.argmax(preds_batch, axis=1))
    y_true.extend(labels_batch.numpy())
    test_images.extend(images_batch.numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_pred_probs = np.array(y_pred_probs)

print(f"✅ Collected predictions for {len(y_true)} test images.")


<a name="accuracy"></a>
## 1️⃣6️⃣ Accuracy

Accuracy is the fraction of correctly classified images out of all images.
Below: the accuracy curve during training (already shown together in Section
14) and the final test-set value in a clear, standalone form.


In [ ]:
final_test_accuracy = (y_pred == y_true).mean()

plt.figure(figsize=(10, 5))
plt.plot(history["accuracy"], label="Train accuracy")
plt.plot(history["val_accuracy"], label="Validation accuracy")
plt.axhline(final_test_accuracy, color="green", linestyle=":", label=f"Test accuracy ({final_test_accuracy:.3f})")
plt.title("Accuracy — train / validation / test")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"🎯 Test accuracy: {final_test_accuracy:.4f} ({final_test_accuracy * 100:.2f}%)")


<a name="loss"></a>
## 1️⃣7️⃣ Loss

The value of the loss function (`sparse_categorical_crossentropy`) — the
lower, the more "confident" the model is in its correct predictions. A large
gap between `loss` (training) and `val_loss` (validation) would signal
overfitting.


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history["loss"], label="Train loss")
plt.plot(history["val_loss"], label="Validation loss")
plt.axhline(test_loss, color="green", linestyle=":", label=f"Test loss ({test_loss:.3f})")
plt.title("Loss — train / validation / test")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"📉 Test loss: {test_loss:.4f}")


<a name="precision"></a>
## 1️⃣8️⃣ Precision

**Precision** answers the question: *of the images the model assigned to a
given class, what fraction actually belonged to it?* High precision matters
when the cost of a "false alarm" (incorrectly detecting a disease that isn't
there) is high — e.g. unnecessary pesticide spraying.

We compute both the **macro** variant (unweighted average across classes —
treats every class equally, regardless of its size) and the **weighted**
variant (accounts for class frequency — better reflects overall performance
on imbalanced data).


In [ ]:
precision_macro = precision_score(y_true, y_pred, average="macro", zero_division=0)
precision_weighted = precision_score(y_true, y_pred, average="weighted", zero_division=0)

print(f"🎯 Precision (macro):    {precision_macro:.4f}")
print(f"🎯 Precision (weighted): {precision_weighted:.4f}")


<a name="recall"></a>
## 1️⃣9️⃣ Recall

**Recall** answers the question: *of all the images that actually belong to
a given class, what fraction did the model correctly identify?* High recall
is critical in this domain — missing a disease (a false negative) is
usually more costly than a false alarm, since an unrecognized disease can
spread across the entire crop.


In [ ]:
recall_macro = recall_score(y_true, y_pred, average="macro", zero_division=0)
recall_weighted = recall_score(y_true, y_pred, average="weighted", zero_division=0)

print(f"🎯 Recall (macro):    {recall_macro:.4f}")
print(f"🎯 Recall (weighted): {recall_weighted:.4f}")


<a name="f1"></a>
## 2️⃣0️⃣ F1-score

**F1-score** is the harmonic mean of precision and recall — a single metric
balancing both aspects. Especially useful for imbalanced data (Section 7),
where accuracy alone can give an overly optimistic picture of model quality
(e.g. if the model only performs well on populous classes).


In [ ]:
f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
f1_weighted = f1_score(y_true, y_pred, average="weighted", zero_division=0)

print(f"🎯 F1-score (macro):    {f1_macro:.4f}")
print(f"🎯 F1-score (weighted): {f1_weighted:.4f}")

metrics_summary = pd.DataFrame({
    "metric": ["Accuracy", "Precision (macro)", "Recall (macro)", "F1-score (macro)"],
    "value": [final_test_accuracy, precision_macro, recall_macro, f1_macro],
})
print("\n", metrics_summary.to_string(index=False))

os.makedirs("results", exist_ok=True)
metrics_summary.to_csv("results/metrics_summary.csv", index=False)
print("\n💾 Metrics summary saved to results/metrics_summary.csv")


<a name="confusion-matrix"></a>
## 2️⃣1️⃣ Confusion Matrix

The confusion matrix shows **exactly which classes are being confused with
each other** — much more detailed information than a single number
(accuracy). With 38 classes the matrix is dense, so we present it as a
heatmap without numeric annotations in the cells (unreadable at this scale)
— color is enough to identify a strong diagonal (correct predictions) and
any "bright spots" off the diagonal (systematic confusion between specific
classes).


In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(16, 14))
sns.heatmap(
    cm,
    cmap="viridis",
    xticklabels=class_names,
    yticklabels=class_names,
    cbar=True,
    square=True,
)
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.title("Confusion Matrix — test set")
plt.xticks(rotation=90, fontsize=6)
plt.yticks(rotation=0, fontsize=6)
plt.tight_layout()
plt.savefig("results/confusion_matrix.png", dpi=150)
plt.show()

print("💾 Confusion matrix saved to results/confusion_matrix.png")

# Top 5 most frequently confused class pairs (off-diagonal)
cm_off_diagonal = cm.copy()
np.fill_diagonal(cm_off_diagonal, 0)
top_confusions_idx = np.dstack(np.unravel_index(np.argsort(-cm_off_diagonal.ravel())[:5], cm.shape))[0]

print("\n🔀 Most frequently confused class pairs:")
for true_idx, pred_idx in top_confusions_idx:
    count = cm[true_idx, pred_idx]
    if count > 0:
        print(f"   {class_names[true_idx]}  →  {class_names[pred_idx]}  ({count} cases)")


<a name="classification-report"></a>
## 2️⃣2️⃣ Classification Report

The full `scikit-learn` report with **per-class** metrics (precision, recall,
f1-score, support) — lets us identify specific classes the model struggles
with most, a valuable pointer for further work (e.g. whether additional
augmentation or more data is needed for a specific disease).


In [ ]:
report_dict = classification_report(
    y_true, y_pred, target_names=class_names, zero_division=0, output_dict=True
)
report_df = pd.DataFrame(report_dict).transpose()

print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

report_df.to_csv("results/classification_report.csv")
print("💾 Full report saved to results/classification_report.csv")

# 5 classes with the lowest F1-score — priority candidates for further analysis
worst_classes = report_df.iloc[:NUM_CLASSES].sort_values("f1-score").head(5)
print("\n⚠️ Classes with the lowest F1-score:")
print(worst_classes[["precision", "recall", "f1-score", "support"]])


<a name="pred-single"></a>
## 2️⃣3️⃣ Single Image Prediction

The helper function `predict_single_image` takes a single image (tensor or
numpy array), runs it through the model, and returns the predicted class
along with the confidence level. This is the basic building block we'll
later use for the full new-image prediction pipeline (Section 28).


In [ ]:
def predict_single_image(model, image, class_names, true_label=None):
    # Predicts the class for a single image and displays the result.
    image_batch = tf.expand_dims(image, axis=0)
    probs = model.predict(image_batch, verbose=0)[0]
    pred_idx = np.argmax(probs)
    confidence = probs[pred_idx]

    plt.figure(figsize=(5, 5))
    plt.imshow(image.numpy().astype("uint8") if hasattr(image, "numpy") else image.astype("uint8"))
    title = f"Prediction: {class_names[pred_idx]}\nConfidence: {confidence * 100:.1f}%"
    if true_label is not None:
        title = f"True: {class_names[true_label]}\n" + title
    plt.title(title, fontsize=10)
    plt.axis("off")
    plt.show()

    return class_names[pred_idx], confidence


# Example usage on a random image from the test set
random_idx = random.randint(0, len(test_images) - 1)
predicted_class, confidence = predict_single_image(
    model, tf.constant(test_images[random_idx]), class_names, true_label=y_true[random_idx]
)


<a name="pred-multi"></a>
## 2️⃣4️⃣ Multiple Image Prediction

A grid visualization of predictions for several images at once — the title
of each image is **green** when the prediction is correct, and **red** when
the model got it wrong. This is a quick way to qualitatively assess model
behavior on a larger sample.


In [ ]:
def plot_prediction_grid(images, y_true_arr, y_pred_arr, y_pred_probs_arr, class_names, n=9):
    indices = random.sample(range(len(images)), n)
    plt.figure(figsize=(14, 14))
    for i, idx in enumerate(indices):
        ax = plt.subplot(int(np.sqrt(n)), int(np.sqrt(n)), i + 1)
        plt.imshow(images[idx].astype("uint8"))
        correct = y_true_arr[idx] == y_pred_arr[idx]
        color = "green" if correct else "red"
        confidence = y_pred_probs_arr[idx][y_pred_arr[idx]] * 100
        plt.title(
            f"Pred: {class_names[y_pred_arr[idx]]}\n({confidence:.0f}%)",
            fontsize=8, color=color,
        )
        plt.axis("off")
    plt.suptitle("Predictions on multiple images (green = correct, red = incorrect)", fontsize=13)
    plt.tight_layout()
    plt.show()


plot_prediction_grid(test_images, y_true, y_pred, y_pred_probs, class_names, n=9)


<a name="bledy"></a>
## 2️⃣5️⃣ Misprediction Visualization

A deliberate look at **specific model mistakes** is one of the most valuable
analysis steps — it helps assess whether the errors are "understandable"
(e.g. two diseases with very similar visual symptoms) or point to a data or
model issue.


In [ ]:
misclassified_idx = np.where(y_true != y_pred)[0]
print(f"❌ Number of mispredictions: {len(misclassified_idx)} / {len(y_true)} "
      f"({len(misclassified_idx) / len(y_true) * 100:.2f}%)")

n_show = min(9, len(misclassified_idx))
sample_errors = np.random.choice(misclassified_idx, n_show, replace=False)

plt.figure(figsize=(14, 14))
for i, idx in enumerate(sample_errors):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(test_images[idx].astype("uint8"))
    confidence = y_pred_probs[idx][y_pred[idx]] * 100
    plt.title(
        f"True: {class_names[y_true[idx]]}\n"
        f"Predicted: {class_names[y_pred[idx]]} ({confidence:.0f}%)",
        fontsize=8, color="red",
    )
    plt.axis("off")
plt.suptitle("Examples of mispredictions", fontsize=13)
plt.tight_layout()
plt.show()


<a name="zapis"></a>
## 2️⃣6️⃣ Saving the Model

We save the final model in Keras's native format (`.keras`) — it contains
the architecture, weights, and optimizer configuration in a single file. We
also save the list of class names in a separate JSON file, since it isn't
automatically saved with the model, and it's required to interpret
prediction results after loading the model in another environment (see
Sections 27–28).


In [ ]:
FINAL_MODEL_PATH = os.path.join(MODELS_DIR, "plant_disease_efficientnetb0_final.keras")
CLASS_NAMES_PATH = os.path.join(MODELS_DIR, "class_names.json")

model.save(FINAL_MODEL_PATH)
print(f"💾 Model saved: {FINAL_MODEL_PATH}")

with open(CLASS_NAMES_PATH, "w", encoding="utf-8") as f:
    json.dump(class_names, f, ensure_ascii=False, indent=2)
print(f"💾 Class names saved: {CLASS_NAMES_PATH}")

model_size_mb = os.path.getsize(FINAL_MODEL_PATH) / (1024 * 1024)
print(f"📦 Model file size: {model_size_mb:.1f} MB")


<a name="wczytanie"></a>
## 2️⃣7️⃣ Loading the Model

We verify that the saved model loads correctly and produces **identical**
results to the model kept in memory — an important test before deployment
(e.g. to a web or mobile application), where the model will be loaded from a
file in a completely new process.


In [ ]:
loaded_model = tf.keras.models.load_model(FINAL_MODEL_PATH)

with open(CLASS_NAMES_PATH, "r", encoding="utf-8") as f:
    loaded_class_names = json.load(f)

# Sanity check: compare the evaluation result of the loaded model with the original
loaded_test_loss, loaded_test_accuracy = loaded_model.evaluate(test_ds, verbose=0)

print(f"✅ Model loaded successfully.")
print(f"   Accuracy (original model): {test_accuracy:.4f}")
print(f"   Accuracy (loaded model):   {loaded_test_accuracy:.4f}")
assert abs(test_accuracy - loaded_test_accuracy) < 1e-4, "Mismatch between the original and loaded model!"
print("✅ Results are identical — the model is ready for deployment.")


<a name="nowe-zdjecia"></a>
## 2️⃣8️⃣ Prediction on New Images

A full "file-to-result" inference pipeline — with automatic saving of the
prediction result (image + label + confidence) to the `predictions/` folder,
simulating how results might be archived in a real application.

**How to actually use this function on your own image — three ways:**

1. **Google Colab (easiest):** run the cell below — a "Choose Files" button
   will appear, letting you pick an image from your computer. It will be
   automatically uploaded to the notebook and classified.
2. **Local Jupyter / any environment:** upload your image to the `images/`
   folder in the repository, then in a new cell call:
   ```python
   predict_new_image("images/my_leaf.jpg", loaded_model, loaded_class_names)
   ```
   (replacing the filename with your own).
3. **Image already present on the server/Colab disk** — simply provide the
   full path to it as the first argument of the function.


In [ ]:
# Directory for saved prediction results. Add it here, locally, so this
# section works standalone without needing to re-run earlier cells.
PREDICTIONS_DIR = "predictions"
os.makedirs(PREDICTIONS_DIR, exist_ok=True)


def predict_new_image(image_path, model, class_names, img_size=IMG_SIZE, top_k=3, save_result=True):
    # Loads an image from disk, returns the `top_k` most probable classes,
    # and optionally saves a visualization of the result to predictions/.
    raw_image = tf.io.read_file(image_path)
    image = tf.io.decode_image(raw_image, channels=3, expand_animations=False)
    image = tf.image.resize(image, (img_size, img_size))
    image = tf.cast(image, tf.float32)

    probs = model.predict(tf.expand_dims(image, axis=0), verbose=0)[0]
    top_indices = np.argsort(probs)[::-1][:top_k]

    plt.figure(figsize=(5, 5))
    plt.imshow(image.numpy().astype("uint8"))
    plt.title(f"Prediction: {class_names[top_indices[0]]}\nConfidence: {probs[top_indices[0]] * 100:.1f}%", fontsize=10)
    plt.axis("off")

    if save_result:
        out_name = f"pred_{pathlib.Path(image_path).stem}_{class_names[top_indices[0]]}.png"
        out_path = os.path.join(PREDICTIONS_DIR, out_name)
        plt.savefig(out_path, dpi=120, bbox_inches="tight")
        print(f"💾 Result saved: {out_path}")

    plt.show()

    print(f"🔝 Top-{top_k} predictions:")
    for idx in top_indices:
        print(f"   {class_names[idx]:45s} {probs[idx] * 100:5.1f}%")

    return [(class_names[idx], float(probs[idx])) for idx in top_indices]


In [ ]:
# ---------------------------------------------------------------
# Interactive upload and classification of your own image.
# In Google Colab, a file-selection button will appear.
# In other environments (local Jupyter) — call the function manually,
# passing the image path (see the example in the comment below).
# ---------------------------------------------------------------
try:
    from google.colab import files

    print("📤 Choose an image from your computer to classify:")
    uploaded = files.upload()

    for fname in uploaded.keys():
        print(f"\n🖼️ Analyzing file: {fname}")
        result = predict_new_image(fname, loaded_model, loaded_class_names)

except ImportError:
    print("ℹ️ Google Colab environment not detected.")
    print("   Upload an image to the images/ folder, for example, then call:")
    print('   predict_new_image("images/my_leaf.jpg", loaded_model, loaded_class_names)')

# Manual call example (uncomment and replace the path):
# result = predict_new_image("images/my_leaf.jpg", loaded_model, loaded_class_names)


<a name="podsumowanie"></a>
## 2️⃣9️⃣ Summary

This project built a complete pipeline for classifying plant diseases from
leaf photographs, using **Transfer Learning** and **Fine Tuning** on the
**EfficientNetB0** architecture:

* Loaded and analyzed the **PlantVillage** dataset (38 classes, ~54,000
  images), including the class distribution and example images.
* Implemented **Data Augmentation** as part of the model's computation graph
  (active only during training).
* Split the data into **train / validation / test** sets (70/15/15%).
* Built the model in two phases: **transfer learning** (frozen base model)
  and **fine-tuning** (last layers unfrozen, low learning rate), with a full
  set of callbacks (`ModelCheckpoint`, `EarlyStopping`, `ReduceLROnPlateau`,
  `TensorBoard`).
* Evaluated the model with multiple metrics: **accuracy, precision, recall,
  F1-score** (macro and weighted), a confusion matrix, and a full per-class
  classification report.
* Implemented functions for single and multiple image predictions, including
  visualization of misclassifications.
* Saved the model in `.keras` format along with the class list, verified
  correct loading, and prepared a ready-to-use inference pipeline for new
  images outside the dataset.

The resulting metrics (accuracy, F1-score) are reported directly in the
cells of Sections 15–22 after training runs — the values depend on the
specific training run and the hardware the notebook is executed on.

<a name="ograniczenia"></a>
## 3️⃣0️⃣ Model Limitations

A fair assessment of this project requires clearly stating its limitations:

* **Laboratory data, not field data.** PlantVillage's photos were taken
  under controlled conditions (uniform background, good lighting, a single
  leaf filling the frame). The model may perform significantly worse on
  photos taken with a phone in the field — with soil, other plants, variable
  lighting, and partially obscured leaves in the background.
* **Risk of shortcut learning.** There's a risk that the model partly bases
  its prediction on background or lighting features characteristic of a
  given batch of photos, rather than solely on disease symptoms — difficult
  to fully rule out without testing on an external dataset.
* **Imbalanced classes** (Section 7) — classes with fewer examples may be
  recognized worse, which is partly confirmed by Section 22 (Classification
  Report).
* **Single leaf, single disease.** The model assumes one label per image —
  it doesn't handle cases where several diseases co-occur on the same leaf,
  nor does it assess infection severity.
* **Limited species range.** The dataset covers a limited number of crop
  species — the model won't generalize to species outside the training set.

<a name="ulepszenia"></a>
## 3️⃣1️⃣ Possible Improvements

* **Field-condition data** — further training or fine-tuning on images taken
  under real-world conditions (e.g. datasets like PlantDoc), possibly using
  domain adaptation techniques.
* **Larger EfficientNet variants** (B3–B7) or newer architectures (ConvNeXt,
  Vision Transformer) — potentially higher accuracy at the cost of greater
  compute requirements.
* **Model explainability (Grad-CAM / Explainable AI)** — visualizing which
  regions of the leaf the model actually focuses on when predicting, which
  would help verify the "shortcut learning" risk mentioned in the
  limitations.
* **Quantization and export to TensorFlow Lite / ONNX** — enabling
  deployment on mobile devices (an offline app for farmers in the field).
* **Deployment as a REST API** (e.g. FastAPI + Docker) exposing the Section
  28 pipeline as a public endpoint.
* **Class weighting or oversampling** for less populous classes, to improve
  metrics for the weakest classes identified in Section 22.
* **Multi-label classification** — handling cases of co-occurring diseases
  or assessing infection severity (regression of symptom intensity), rather
  than a single class label.
* **Continuous production monitoring** (e.g. using MLflow) — tracking data
  drift and prediction quality over time after deployment.

---

## 🛠️ Tech Stack

`Python` · `TensorFlow / Keras` · `EfficientNetB0` · `Kaggle API` ·
`scikit-learn` · `NumPy` · `Pandas` · `Matplotlib` · `Seaborn` · `TensorBoard`

## 👤 Author

AIResearchForge — a TensorFlow/Keras-based project for image classification, applying Transfer Learning and Fine-Tuning to solve a real-world agricultural problem.

## 📄 License

This project is released under the MIT license (see the `LICENSE` file in
the repository). The PlantVillage dataset is subject to the license set by
its authors — see
[Kaggle — PlantVillage Dataset](https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset).
